# Telecom Customer Churn — Synthetic Data Generator
**Use Case:** Customer Churn Prediction (Telecommunications / ISP)  
**Target users:** Customer Value Management Analysts, Data Scientists

This notebook generates a realistic synthetic panel dataset that mirrors real-world telecom customer behaviour. It follows the structural model defined in the *Model Research & Specifications* document.

---
### What gets generated
| Table | Rows (1 000 customers, 12 months) |
|---|---|
| `customers` | 1 000 |
| `subscriptions` | 1 000 |
| `usage_monthly` | 12 000 |
| `support_tickets` | ~12 000 |
| `payments` | 12 000 |

> **Latent variables** (`CSAT`, `propensity_to_churn`, `payment_reliability`, `tech_savviness`) drive the data-generating process but are **dropped** from the final export.


## 0 · Setup & Imports

In [1]:
# Import libraries
import numpy as np
import pandas as pd
from scipy.special import expit          # sigmoid / logistic function
import warnings, os
warnings.filterwarnings("ignore")

# Reproducibility
RNG_SEED = 42
rng = np.random.default_rng(RNG_SEED)

## 1 · Configuration

Tweak these knobs to change the size and characteristics of the dataset.


In [2]:
# Dataset size
N_CUSTOMERS = 1_000          # number of unique customers
N_MONTHS    = 12             # observation window (months)

# Plan definitions
PLAN_NAMES      = ["Basic", "Standard", "Premium"]
PLAN_PROBS      = [0.50, 0.30, 0.20]          # Categorical probabilities
PLAN_BASE_FEES  = {"Basic": 299, "Standard": 599, "Premium": 999}   # PHP
PLAN_MULTIPLIER = {"Basic": 0.5, "Standard": 1.0, "Premium": 1.8}  # usage

# Churn model coefficients
CHURN_INTERCEPT   = -1.5
CHURN_TENURE      = -0.02
CHURN_CSAT        = -2.0
CHURN_PROPENSITY  =  1.5
CHURN_TICKETS     =  0.3

# Realism / noise
MISSING_USAGE_RATE  = 0.05   # 5 % of usage rows get NaN for data_usage_gb
MISSING_INCOME_RATE = 0.03   # 3 % of customers get NaN for income_bracket
OUTLIER_TOP_PCT     = 0.01   # top 1 % usage inflated ×3

print("Configuration set")
print(f"    Customers : {N_CUSTOMERS:,}")
print(f"    Months    : {N_MONTHS}")
print(f"    Total rows: {N_CUSTOMERS * N_MONTHS:,} (usage / payments)")


Configuration set
    Customers : 1,000
    Months    : 12
    Total rows: 12,000 (usage / payments)


## 2 · Latent Variables

These hidden drivers shape every observable feature.  
They are **removed** before final export.

| Variable | Distribution | Role |
|---|---|---|
| CSAT | Beta(2, 2) | Customer satisfaction → tickets, churn |
| propensity_to_churn | Beta(1, 3) | Switching tendency → tenure, churn |
| payment_reliability | Beta(3, 1) | On-time payment likelihood → delay |
| tech_savviness | Uniform(0, 1) | Digital fluency → usage, tickets |


In [3]:
CSAT               = rng.beta(2, 2, N_CUSTOMERS)          # bounded [0,1]
propensity_to_churn = rng.beta(1, 3, N_CUSTOMERS)          # skewed → low churn
payment_reliability = rng.beta(3, 1, N_CUSTOMERS)          # skewed → reliable
tech_savviness      = rng.uniform(0, 1, N_CUSTOMERS)

print("Latent variable summaries:")
for name, arr in [
    ("CSAT",                CSAT),
    ("propensity_to_churn", propensity_to_churn),
    ("payment_reliability", payment_reliability),
    ("tech_savviness",      tech_savviness),
]:
    print(f"  {name:<24} mean={arr.mean():.3f}  std={arr.std():.3f}"
          f"  min={arr.min():.3f}  max={arr.max():.3f}")


Latent variable summaries:
  CSAT                     mean=0.503  std=0.223  min=0.012  max=0.980
  propensity_to_churn      mean=0.240  std=0.188  min=0.001  max=0.915
  payment_reliability      mean=0.755  std=0.188  min=0.077  max=1.000
  tech_savviness           mean=0.514  std=0.285  min=0.001  max=0.999


## 3 · `customers` Table

In [4]:
customer_ids = np.arange(1, N_CUSTOMERS + 1)

age    = rng.normal(35, 10, N_CUSTOMERS).clip(18, 80).astype(int)
gender = rng.choice(["Male", "Female"], N_CUSTOMERS)
region = rng.choice(["NCR", "Luzon", "Visayas", "Mindanao"], N_CUSTOMERS,
                     p=[0.40, 0.30, 0.18, 0.12])
income = rng.choice(["Low", "Mid", "High"], N_CUSTOMERS, p=[0.40, 0.40, 0.20])
channel = rng.choice(["Web", "App", "Store"], N_CUSTOMERS, p=[0.45, 0.35, 0.20])

customers = pd.DataFrame({
    "customer_id"    : customer_ids,
    "age"            : age,
    "gender"         : gender,
    "region"         : region,
    "income_bracket" : income,
    "signup_channel" : channel,
})

# Inject 3% missing income
missing_mask = rng.random(N_CUSTOMERS) < MISSING_INCOME_RATE
customers.loc[missing_mask, "income_bracket"] = np.nan

print(f"customers shape : {customers.shape}")
customers.head()


customers shape : (1000, 6)


,customer_id,age,gender,region,income_bracket,signup_channel
0,1,33,Male,Luzon,Mid,App
1,2,35,Male,Mindanao,Low,Web
2,3,33,Female,Luzon,Mid,App
3,4,46,Male,NCR,Low,Web
4,5,46,Female,NCR,Mid,Web


## 4 · `subscriptions` Table

In [5]:
plan_type = rng.choice(PLAN_NAMES, N_CUSTOMERS, p=PLAN_PROBS)
base_fees = np.array([PLAN_BASE_FEES[p] for p in plan_type], dtype=float)

monthly_fee = (base_fees + rng.normal(0, 50, N_CUSTOMERS)).clip(0)
contract    = rng.choice(["Prepaid", "Postpaid"], N_CUSTOMERS, p=[0.40, 0.60])
discount    = rng.choice([0.0, 0.05, 0.10, 0.15], N_CUSTOMERS,
                          p=[0.55, 0.20, 0.15, 0.10])

subscriptions = pd.DataFrame({
    "subscription_id" : np.arange(1, N_CUSTOMERS + 1),
    "customer_id"     : customer_ids,
    "plan_type"       : plan_type,
    "monthly_fee"     : monthly_fee.round(2),
    "contract_type"   : contract,
    "discount_pct"    : discount,
})

print(f"subscriptions shape : {subscriptions.shape}")
print("\nPlan distribution:")
print(subscriptions["plan_type"].value_counts())
subscriptions.head()


subscriptions shape : (1000, 6)

Plan distribution:
plan_type
Basic       492
Standard    300
Premium     208
Name: count, dtype: int64


,subscription_id,customer_id,plan_type,monthly_fee,contract_type,discount_pct
0,1,1,Basic,287.96,Postpaid,0.15
1,2,2,Basic,329.86,Prepaid,0.15
2,3,3,Basic,313.36,Postpaid,0.00
3,4,4,Basic,326.43,Postpaid,0.00
4,5,5,Standard,496.70,Postpaid,0.10


## 5 · Tenure

$$\text{tenure}_i \sim \mathrm{Poisson}(\lambda = 12 + 24\cdot(1 - \text{propensity}_i))$$

Higher loyalty (low propensity) → longer tenure.


In [6]:
tenure_lambda = 12 + 24 * (1 - propensity_to_churn)
tenure = rng.poisson(tenure_lambda).clip(1)          # at least 1 month

print(f"Tenure  mean={tenure.mean():.1f}  median={np.median(tenure):.0f}"
      f"  min={tenure.min()}  max={tenure.max()}")


Tenure  mean=30.5  median=30  min=8  max=52


## 6 · `usage_monthly` Table

In [7]:
plan_mult = np.array([PLAN_MULTIPLIER[p] for p in plan_type])

rows_usage = []
for t in range(1, N_MONTHS + 1):
    mu = 5 + 10 * plan_mult + 5 * tech_savviness
    data_gb   = rng.normal(mu, 4).clip(0)
    calls     = rng.poisson(100 + 50 * (1 - tech_savviness))
    sms       = rng.poisson(50, N_CUSTOMERS)
    roaming   = rng.exponential(0.5, N_CUSTOMERS).clip(0)

    rows_usage.append(pd.DataFrame({
        "usage_id"      : np.arange(1, N_CUSTOMERS + 1) + (t - 1) * N_CUSTOMERS,
        "customer_id"   : customer_ids,
        "month"         : t,
        "data_usage_gb" : data_gb.round(3),
        "call_minutes"  : calls,
        "sms_count"     : sms,
        "roaming_usage" : roaming.round(3),
    }))

usage_monthly = pd.concat(rows_usage, ignore_index=True)

# Inject 5% missing data_usage_gb
missing_mask = rng.random(len(usage_monthly)) < MISSING_USAGE_RATE
usage_monthly.loc[missing_mask, "data_usage_gb"] = np.nan

# Inject outliers: top 1% users × 3
threshold = usage_monthly["data_usage_gb"].quantile(1 - OUTLIER_TOP_PCT)
outlier_mask = usage_monthly["data_usage_gb"] > threshold
usage_monthly.loc[outlier_mask, "data_usage_gb"] *= 3

# call_minutes outliers × 2
call_threshold = usage_monthly["call_minutes"].quantile(1 - OUTLIER_TOP_PCT)
call_outlier   = usage_monthly["call_minutes"] > call_threshold
usage_monthly.loc[call_outlier, "call_minutes"] = (
    usage_monthly.loc[call_outlier, "call_minutes"] * 2
)

print(f"usage_monthly shape : {usage_monthly.shape}")
print(f"Missing data_usage_gb : {usage_monthly['data_usage_gb'].isna().sum():,}"
      f" ({usage_monthly['data_usage_gb'].isna().mean():.1%})")
usage_monthly.head()


usage_monthly shape : (12000, 7)
Missing data_usage_gb : 593 (4.9%)


,usage_id,customer_id,month,data_usage_gb,call_minutes,sms_count,roaming_usage
0,1,1,1,17.049,104,56,0.121
1,2,2,1,8.184,142,45,0.045
2,3,3,1,13.870,141,45,0.147
3,4,4,1,17.335,75,44,0.364
4,5,5,1,9.878,118,44,0.585


## 7 · `support_tickets` Table

$$\lambda_{i,t} = 2 + 3(1-\text{CSAT}_i) + (1-\text{tech}_i)$$
$$\text{tickets}_{i,t} \sim \mathrm{Poisson}(\lambda_{i,t})$$


In [8]:
issue_types = ["Billing", "Technical", "Service"]
rows_tickets = []
ticket_id = 1

lambda_tickets = 2 + 3 * (1 - CSAT) + (1 - tech_savviness)

for t in range(1, N_MONTHS + 1):
    counts = rng.poisson(lambda_tickets)
    issues = rng.choice(issue_types, N_CUSTOMERS, p=[0.35, 0.40, 0.25])

    rows_tickets.append(pd.DataFrame({
        "ticket_id"    : np.arange(ticket_id, ticket_id + N_CUSTOMERS),
        "customer_id"  : customer_ids,
        "month"        : t,
        "ticket_count" : counts,
        "issue_type"   : issues,
    }))
    ticket_id += N_CUSTOMERS

support_tickets = pd.concat(rows_tickets, ignore_index=True)

print(f"support_tickets shape : {support_tickets.shape}")
print(f"Mean tickets / customer / month : {support_tickets['ticket_count'].mean():.2f}")
print("\nIssue type distribution:")
print(support_tickets["issue_type"].value_counts())
support_tickets.head()


support_tickets shape : (12000, 5)
Mean tickets / customer / month : 3.96

Issue type distribution:
issue_type
Technical    4719
Billing      4231
Service      3050
Name: count, dtype: int64


,ticket_id,customer_id,month,ticket_count,issue_type
0,1,1,1,6,Service
1,2,2,1,3,Technical
2,3,3,1,2,Technical
3,4,4,1,4,Technical
4,5,5,1,4,Technical


## 8 · `payments` Table

In [9]:
rows_payments = []

for t in range(1, N_MONTHS + 1):
    amount_due  = (monthly_fee * (1 - discount)).round(2)
    amount_paid = (amount_due + rng.normal(0, 20, N_CUSTOMERS)).clip(0).round(2)

    # Exponential delay; unreliable payers → higher expected delay
    delay = rng.exponential(5 * (1 - payment_reliability + 0.05))
    delay = delay.clip(0).round(1)   # clip negatives (logging artefact)

    rows_payments.append(pd.DataFrame({
        "payment_id"    : np.arange(1, N_CUSTOMERS + 1) + (t - 1) * N_CUSTOMERS,
        "customer_id"   : customer_ids,
        "billing_month" : t,
        "amount_due"    : amount_due,
        "amount_paid"   : amount_paid,
        "delay_days"    : delay,
    }))

payments = pd.concat(rows_payments, ignore_index=True)

print(f"payments shape : {payments.shape}")
print(f"Mean delay_days : {payments['delay_days'].mean():.2f}")
print(f"Pct on-time (delay=0) : {(payments['delay_days'] == 0).mean():.1%}")
payments.head()


payments shape : (12000, 6)
Mean delay_days : 1.47
Pct on-time (delay=0) : 5.0%


,payment_id,customer_id,billing_month,amount_due,amount_paid,delay_days
0,1,1,1,244.77,268.02,2.4
1,2,2,1,280.38,268.93,1.1
2,3,3,1,313.36,324.08,0.7
3,4,4,1,326.43,319.97,2.1
4,5,5,1,447.03,449.06,1.8


## 9 · Churn Model

$$P(\text{churn}_i = 1) = \sigma\!\left(
  -3
  - 0.02\cdot\text{tenure}_i
  - 2\cdot\text{CSAT}_i
  + 1.5\cdot\text{propensity}_i
  + 0.3\cdot\overline{\text{tickets}}_i
\right)$$


In [10]:
# Aggregate: mean ticket count per customer across all months
avg_tickets = (
    support_tickets.groupby("customer_id")["ticket_count"]
    .mean()
    .values
)

logit_p = (
    CHURN_INTERCEPT
    + CHURN_TENURE     * tenure
    + CHURN_CSAT       * CSAT
    + CHURN_PROPENSITY * propensity_to_churn
    + CHURN_TICKETS    * avg_tickets
)

churn_prob = expit(logit_p)
churn      = rng.binomial(1, churn_prob).astype(bool)

print(f"Churn rate : {churn.mean():.2%}  (target: 15–25%)")
print(f"Churned    : {churn.sum():,} / {N_CUSTOMERS:,}")

# Quick sanity check — average churn probability
import pandas as _pd
_diag = _pd.DataFrame({
    "churn_prob": churn_prob,
    "plan"      : plan_type,
    "churned"   : churn,
})
print("\nChurn rate by plan:")
print(_diag.groupby("plan")["churned"].mean().round(4))


Churn rate : 18.40%  (target: 15–25%)
Churned    : 184 / 1,000

Churn rate by plan:
plan
Basic       0.1992
Premium     0.1779
Standard    0.1633
Name: churned, dtype: float64


## 10 · Attach Churn Label to `customers`

In [11]:
customers["tenure_months"] = tenure
customers["churned"]       = churn

print(f"customers shape : {customers.shape}")
print("\nChurn by income bracket:")
print(customers.groupby("income_bracket")["churned"].mean().round(4))
customers.head()


customers shape : (1000, 8)

Churn by income bracket:
income_bracket
High    0.1613
Low     0.1744
Mid     0.2000
Name: churned, dtype: float64


,customer_id,age,gender,region,income_bracket,signup_channel,tenure_months,churned
0,1,33,Male,Luzon,Mid,App,39,False
1,2,35,Male,Mindanao,Low,Web,15,False
2,3,33,Female,Luzon,Mid,App,33,False
3,4,46,Male,NCR,Low,Web,15,False
4,5,46,Female,NCR,Mid,Web,34,False


## 11 · Data Quality Checks

In [12]:
print("=" * 60)
print("DATASET SUMMARY")
print("=" * 60)

tables = {
    "customers"      : customers,
    "subscriptions"  : subscriptions,
    "usage_monthly"  : usage_monthly,
    "support_tickets": support_tickets,
    "payments"       : payments,
}

for name, df in tables.items():
    missing = df.isnull().sum().sum()
    pct     = missing / df.size
    print(f"  {name:<20} shape={str(df.shape):<18} "
          f"missing={missing:,} ({pct:.2%})")

print()
print("Churn class balance:")
vc = customers["churned"].value_counts(normalize=True)
print(f"  Not churned : {vc[False]:.2%}")
print(f"  Churned     : {vc[True]:.2%}")

print()
print("Usage stats (data_usage_gb):")
print(usage_monthly["data_usage_gb"].describe().round(2))


DATASET SUMMARY
  customers            shape=(1000, 8)          missing=29 (0.36%)
  subscriptions        shape=(1000, 6)          missing=0 (0.00%)
  usage_monthly        shape=(12000, 7)         missing=593 (0.71%)
  support_tickets      shape=(12000, 5)         missing=0 (0.00%)
  payments             shape=(12000, 6)         missing=0 (0.00%)

Churn class balance:
  Not churned : 81.60%
  Churned     : 18.40%

Usage stats (data_usage_gb):
count    11407.00
mean        17.47
std         10.78
min          0.00
25%         11.98
50%         16.02
75%         21.20
max        118.82
Name: data_usage_gb, dtype: float64


## 12 · Export to CSV

All 5 tables are saved to a `synthetic_telecom_data/` folder.  
Latent variables (`CSAT`, `propensity_to_churn`, `payment_reliability`, `tech_savviness`) are **not** included in any exported file.


In [13]:
OUT_DIR = "clean_data"
os.makedirs(OUT_DIR, exist_ok=True)

export_map = {
    "customers.csv"       : customers,
    "subscriptions.csv"   : subscriptions,
    "usage_monthly.csv"   : usage_monthly,
    "support_tickets.csv" : support_tickets,
    "payments.csv"        : payments,
}

for fname, df in export_map.items():
    path = os.path.join(OUT_DIR, fname)
    df.to_csv(path, index=False)
    print(f"{fname:<28} {df.shape[0]:>8,} rows  →  {path}")

print(f"\n All files saved to ./{OUT_DIR}/")


customers.csv                   1,000 rows  →  clean_data/customers.csv
subscriptions.csv               1,000 rows  →  clean_data/subscriptions.csv
usage_monthly.csv              12,000 rows  →  clean_data/usage_monthly.csv
support_tickets.csv            12,000 rows  →  clean_data/support_tickets.csv
payments.csv                   12,000 rows  →  clean_data/payments.csv

 All files saved to ./clean_data/
